[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/06_series/06_1_Labels_and_Lookups.ipynb)

# 06.1: Labels and Lookups

So far you have stored related data in lists. If you had a list of college majors and a separate list of the typical salary for each major, you would rely on position to connect them: the salary at position 0 belongs to the major at position 0. This notebook shows where that approach breaks down, and it introduces the pandas **Series**, which keeps each value together with a label so that the connection cannot be lost. By the end you will be able to create a series and look up values in it by label and by position.

## The data

We will use data on recent college graduates in 173 majors, collected from U.S. Census data around 2010-2012. The next cell uses a library called **pandas** to read the file and pull out two lists. `import pandas as pd` loads the library and gives it the short name `pd`, which is what nearly everyone calls it. You will learn how `read_csv` works in module 07; for now, all that matters is that you end up with two ordinary Python lists.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
median_list = majors["Median"].to_list()

In [ ]:
print(major_list[:3])
print(median_list[:3])
print(len(major_list), len(median_list))

`major_list` holds the names and `median_list` holds the median salary in dollars for full-time workers with that major. The **median** is the middle value: half of the workers earned more and half earned less. The first major is PETROLEUM ENGINEERING with a median of 110000, and both lists have 173 items.

## The problem with parallel lists

Lists like these are called parallel lists, because the only thing connecting them is that matching items sit at the same position. Suppose you want the salaries in order, and you sort `median_list`. Here we sort a copy so that the original list is not changed.

In [ ]:
sorted_medians = sorted(median_list)
print(major_list[0], sorted_medians[0])

This now pairs PETROLEUM ENGINEERING with 22000, which is wrong: the real value is 110000, and 22000 belongs to LIBRARY SCIENCE. Python did not raise an error, because it has no way to know that the two lists were connected. Every answer you computed from these lists after that point would be silently wrong.

## What is a series?

A **Series** is a pandas object that stores a column of values, with a label attached to each value. The labels are called the **index**. To create one, give `pd.Series()` the values, and use the `index=` argument for the labels.

In [ ]:
salaries = pd.Series(median_list, index=major_list)
salaries

pandas shows two columns. The left one is the index (the major names) and the right one is the values (the salaries). The last line gives the length, 173, and the **dtype**, which is the data type of the values: `int64` is pandas' name for integers. pandas also leaves out the middle rows, so that a long series does not fill your screen. The type of this object is not a list.

In [ ]:
type(salaries)

Now we can repeat the sort. `.sort_values()` returns a new series in order of its values.

In [ ]:
salaries.sort_values()

The top row is now LIBRARY SCIENCE at 22000, and PETROLEUM ENGINEERING with 110000 is at the bottom. Each salary moved together with its label, which is the problem from the previous section solved. Sorting did not change `salaries`, as the next cell shows; the sorted series is a new result that we did not store. You will learn more about this in notebook 06.2.

In [ ]:
salaries.head(1)

## A series without labels

If you do not supply an index, pandas numbers the values starting at 0. These numbers are still labels, and they stay with their values just as the major names did.

In [ ]:
quiz_scores = pd.Series([88, 95, 71])
quiz_scores

## Looking at part of a series

A series can be too long to read in one piece. `.head()` shows the first five items and `.tail()` shows the last five. Each one takes an optional number if you want more or fewer. `len()` works the same way it does on a list.

In [ ]:
salaries.head()

In [ ]:
salaries.tail(3)

In [ ]:
len(salaries)

The file happens to be ordered from highest to lowest salary, which is why `.head()` shows engineering majors and `.tail(3)` shows CLINICAL PSYCHOLOGY, COUNSELING PSYCHOLOGY and LIBRARY SCIENCE. You can also get at the two parts of a series separately with `.index` and `.values`. There are no parentheses after these two, because they are stored attributes of the series and not methods.

In [ ]:
salaries.index

In [ ]:
salaries.values

## Looking up a value by label: `.loc`

To get one value by its label, use `.loc` with square brackets. Think of it as asking "what is stored under this label?"

In [ ]:
salaries.loc["NURSING"]

The result is a single number, 48000, and not a series. In Colab it may be displayed as `np.int64(48000)`. That is how pandas' underlying number library shows an integer, and it behaves like the ordinary integer 48000. To look up several labels at once, pass a list of labels, which does give back a series.

In [ ]:
salaries.loc[["NURSING", "ECONOMICS", "STUDIO ARTS"]]

NURSING is 48000, ECONOMICS is 47000 and STUDIO ARTS is 29000, in the order we asked for them. If a label does not exist, for example because of a misspelling, pandas raises a `KeyError`. This is a useful error to recognize, because it tells you that the label you typed is not in the index. Labels must match exactly, including upper and lower case. The next cell fails on purpose. If you use Run all, Colab stops at this cell, so run the cells after it yourself.

In [ ]:
salaries.loc["Nursing"]

## Looking up a value by position: `.iloc`

Sometimes you care about position and not about the label. `.iloc` works like indexing a list: it starts at 0, and negative numbers count from the end. The `i` stands for integer.

In [ ]:
salaries.iloc[0]

In [ ]:
salaries.iloc[-1]

Position 0 gives 110000 (PETROLEUM ENGINEERING) and position -1 gives 22000 (LIBRARY SCIENCE). Using `.iloc` depends on the current order of the series. If the series were sorted differently, position 0 would hold something else, while `.loc["NURSING"]` would still return the value for nursing. So prefer `.loc` whenever you know the label.

## Slicing

Both `.loc` and `.iloc` can take a slice, which returns a shorter series. With `.iloc`, a slice works the way it does on a list: the stop position is **not** included.

In [ ]:
salaries.iloc[10:13]

Positions 10, 11 and 12 give three items: COMPUTER ENGINEERING, AEROSPACE ENGINEERING and BIOMEDICAL ENGINEERING, all at 60000. With `.loc`, you slice from one label to another, and the stop label **is** included. It has to be, because you may not know which label comes right after the one you want to stop at.

In [ ]:
salaries.loc["COMPUTER ENGINEERING":"BIOMEDICAL ENGINEERING"]

This returns the same three items, including BIOMEDICAL ENGINEERING. Leaving off the stop label continues to the end of the series, so `salaries.loc["CLINICAL PSYCHOLOGY":]` would return the last three items.

## What's next

You can now create a series and pull out specific values by label or by position. Looking things up one at a time only helps when you already know what you want, though. Notebook 06.2 shows how to work on all of the values at once without a loop, which lets you ask questions like "which majors have a median above $70,000?"